In [1]:
import pathlib
import logging

# --- EDIT THESE PATHS ---
TIFF = "/mnt/jwh83-data/Confetti/data/CODEX/new/done/B004-A-404.tif"
SEG_MASK = "/mnt/jwh83-data/Confetti/output/intestine_data_mask_outputs/CellSAM_3_20260113_TIF/B004-A-404_rgb_cellSAM.tif"
MARKERS_CSV = "/mnt/jwh83-data/Confetti/data/CODEX/ns_mp_codex_channel_list.csv"
OUTDIR = "/mnt/jwh83-data/Confetti/output/Redsea/20260127_ns444_CellSAM/B004-A-404"

# optional:
ELEMENT_SHAPE = "star"   # "star" or "square"
ELEMENT_SIZE  = 2        # your CLI used 1
MARKERS_OF_INTEREST = None  # e.g. ["CD3", "CD4"] or None for all

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s: %(message)s")

In [2]:
import numpy as np
import pandas as pd
import skimage
from skimage.io import imread
import skimage.measure
import skimage.morphology

import redseapy.redsea as rs  # uses your installed package


def run_redsea_fixed(
    tiff,
    seg_mask,
    markers_csv,
    output_dir,
    markers_of_interest=None,
    boundary_mode=2,  # ignored in original too
    compensation_mode=1,
    element_shape=2,  # 2=star/diamond, 1=square
    element_size=2,
):
    # --- Helper: same as upstream ---
    def ismember(a, b):
        bind = {}
        for i, elt in enumerate(b):
            if elt not in bind:
                bind[elt] = i
        return [bind.get(itm, None) for itm in a]

    def printProgressBar(iteration, total, prefix="", suffix="", decimals=1, length=50, fill="█", printEnd="\r"):
        percent = ("{0:." + str(decimals) + "f}").format(100 * (iteration / float(total)))
        filledLength = int(length * iteration // total) if total else length
        bar = fill * filledLength + "-" * (length - filledLength)
        print(f"\r{prefix} |{bar}| {percent}% {suffix}", end=printEnd)
        if iteration == total:
            print()

    massDS = pd.read_csv(markers_csv)
    if "marker_name" not in massDS.columns:
        massDS = pd.read_csv(markers_csv, header=None)
        massDS.columns = ["marker_name"]

    # remove wavelength suffixes (same as upstream)
    markers = []
    for m in massDS.marker_name:
        for s in ["_488", "_555", "_570", "_647", "_660"]:
            m = m.replace(s, "")
        markers.append(m)
    massDS["marker_name"] = markers

    if markers_of_interest is None:
        markers_of_interest = markers

    normChannels = markers_of_interest
    normChannelsInds = ismember(normChannels, massDS["marker_name"])
    channelNormIdentity = np.zeros((len(massDS["marker_name"]), 1))
    for i in range(len(normChannelsInds)):
        if normChannelsInds[i] is not None:
            channelNormIdentity[normChannelsInds[i]] = 1

    clusterChannels = massDS["marker_name"]
    clusterChannelsInds = np.where(np.isin(clusterChannels, massDS["marker_name"]))[0]

    logging.info("Reading image")
    tiff_img = imread(tiff)
    countsNoNoise = np.swapaxes(np.swapaxes(tiff_img, 0, 1), 1, 2)

    logging.info("Reading segmentation mask")
    segMat = imread(seg_mask)

    # relabel sequentially (same as upstream)
    lb = 0
    for i in sorted(np.unique(segMat)):
        segMat = np.where(segMat == i, lb, segMat)
        lb += 1

    logging.info("Quantifying markers before correction")
    labelNum = int(np.max(segMat))
    stats = skimage.measure.regionprops(segMat)
    newLmod = segMat

    channelNum = len(clusterChannels)
    data = np.zeros((labelNum, channelNum))
    dataScaleSize = np.zeros((labelNum, channelNum))
    cellSizes = np.zeros((labelNum, 1))

    centroid_xs = []
    centroid_ys = []

    for i in range(labelNum):
        label_counts = [countsNoNoise[coord[0], coord[1], :] for coord in stats[i].coords]
        data[i, 0:channelNum] = np.sum(label_counts, axis=0)
        dataScaleSize[i, 0:channelNum] = np.sum(label_counts, axis=0) / stats[i].area
        cellSizes[i] = stats[i].area
        centroid_xs.append(stats[i].centroid[0])
        centroid_ys.append(stats[i].centroid[1])

    [rowNum, colNum] = newLmod.shape
    cellNum = labelNum

    cellPairMap = np.zeros((cellNum, cellNum))
    newLmod_border = np.pad(newLmod, pad_width=1, mode="constant", constant_values=0)

    logging.info("Creating cell-cell contact matrix")
    for i in range(rowNum):
        for j in range(colNum):
            if newLmod[i, j] == 0:
                tempMatrix = newLmod_border[i : i + 3, j : j + 3]
                tempFactors = np.unique(tempMatrix)
                tempFactors = tempFactors - 1
                if len(tempFactors) == 3:
                    cellPairMap[tempFactors[1], tempFactors[2]] += 1
                elif len(tempFactors) == 4:
                    cellPairMap[tempFactors[1], tempFactors[2]] += 1
                    cellPairMap[tempFactors[1], tempFactors[3]] += 1
                    cellPairMap[tempFactors[2], tempFactors[3]] += 1
                elif len(tempFactors) == 5:
                    cellPairMap[tempFactors[1], tempFactors[2]] += 1
                    cellPairMap[tempFactors[1], tempFactors[3]] += 1
                    cellPairMap[tempFactors[1], tempFactors[4]] += 1
                    cellPairMap[tempFactors[2], tempFactors[3]] += 1
                    cellPairMap[tempFactors[2], tempFactors[4]] += 1
                    cellPairMap[tempFactors[3], tempFactors[4]] += 1

    cellPairMap = cellPairMap + np.transpose(cellPairMap)
    cellBoundaryTotal = np.sum(cellPairMap, axis=0)

    # remove cells without neighbors (same as upstream)
    no_neighbor_cells = np.where(cellBoundaryTotal == 0)[0]
    if len(no_neighbor_cells) > 0:
        cellPairMap = np.delete(cellPairMap, no_neighbor_cells, axis=0)
        cellPairMap = np.delete(cellPairMap, no_neighbor_cells, axis=1)
        cellNum = cellNum - len(no_neighbor_cells)
        labelNum = cellNum
        cellBoundaryTotal = np.delete(cellBoundaryTotal, no_neighbor_cells, axis=0)
        data = np.delete(data, no_neighbor_cells, axis=0)
        dataScaleSize = np.delete(dataScaleSize, no_neighbor_cells, axis=0)
        cellSizes = np.delete(cellSizes, no_neighbor_cells, axis=0)
        centroid_xs = np.delete(np.array(centroid_xs), no_neighbor_cells, axis=0)
        centroid_ys = np.delete(np.array(centroid_ys), no_neighbor_cells, axis=0)
    else:
        centroid_xs = np.array(centroid_xs)
        centroid_ys = np.array(centroid_ys)

    cellBoundaryTotalMatrix = np.tile(cellBoundaryTotal, (cellNum, 1))
    cellPairNorm = compensation_mode * np.identity(cellNum) - cellPairMap / cellBoundaryTotalMatrix
    cellPairNorm = np.transpose(cellPairNorm)

    MIBIdataNearEdge1 = np.zeros((cellNum, channelNum))

    logging.info("Performing correction")
    items = list(range(cellNum))
    l = len(items)
    printProgressBar(0, l, prefix="Progress:", suffix="Complete", length=50)

    if element_shape == 1:  # square
        square = skimage.morphology.square(2 * element_size + 1)
        square_loc = np.where(square == 1)
    elif element_shape == 2:  # "star"/diamond in this implementation
        diam = skimage.morphology.diamond(element_size)
        diam_loc = np.where(diam == 1)
    else:
        raise ValueError("element_shape not recognized")

    for i in range(cellNum):
        label = i + 1
        [tempRow, tempCol] = np.where(newLmod == label)

        for j in range(len(tempRow)):
            label_in_shape = []
            if (
                (element_size - 1 < tempRow[j])
                and (tempRow[j] < rowNum - element_size - 2)
                and (element_size - 1 < tempCol[j])
                and (tempCol[j] < colNum - element_size - 2)
            ):
                ini_point = [tempRow[j] - element_size, tempCol[j] - element_size]

                if element_shape == 1:
                    square_loc_ini_x = [item + ini_point[0] for item in square_loc[0]]
                    square_loc_ini_y = [item + ini_point[1] for item in square_loc[1]]
                    label_in_shape = [
                        newLmod[square_loc_ini_x[k], square_loc_ini_y[k]]
                        for k in range(len(square_loc_ini_x))
                    ]
                else:
                    diam_loc_ini_x = [item + ini_point[0] for item in diam_loc[0]]
                    diam_loc_ini_y = [item + ini_point[1] for item in diam_loc[1]]
                    label_in_shape = [
                        newLmod[diam_loc_ini_x[k], diam_loc_ini_y[k]]
                        for k in range(len(diam_loc_ini_x))
                    ]

            if 0 in label_in_shape:
                MIBIdataNearEdge1[i, :] = MIBIdataNearEdge1[i, :] + countsNoNoise[tempRow[j], tempCol[j], :]

        printProgressBar(i + 1, l, prefix="Progress:", suffix="Complete", length=50)

    MIBIdataNorm2 = np.transpose(np.dot(np.transpose(MIBIdataNearEdge1), cellPairNorm))
    MIBIdataNorm2 = MIBIdataNorm2 + data
    MIBIdataNorm2[MIBIdataNorm2 < 0] = 0

    rev_channelNormIdentity = np.ones_like(channelNormIdentity) - channelNormIdentity
    MIBIdataNorm2 = (
        data * np.transpose(np.tile(rev_channelNormIdentity, (1, cellNum)))
        + MIBIdataNorm2 * np.transpose(np.tile(channelNormIdentity, (1, cellNum)))
    )

    dataCompenScaleSize = MIBIdataNorm2 / cellSizes

    # --- This filter exists in upstream ---
    labelIdentityNew2 = np.ones(cellNum)
    sumDataScaleSizeInClusterChannels = np.sum(dataScaleSize[:, clusterChannelsInds], axis=1)
    labelIdentityNew2[sumDataScaleSizeInClusterChannels < 0.1] = 2

    keep = (labelIdentityNew2 == 1)

    dataCells = data[keep, :]
    dataScaleSizeCells = dataScaleSize[keep, :]
    dataCompenCells = MIBIdataNorm2[keep, :]
    dataCompenScaleSizeCells = dataCompenScaleSize[keep, :]

    labelVec = np.where(keep)
    labelVec = [item + 1 for item in labelVec]  # keep upstream behavior

    cellSizesVec = cellSizes[keep].flatten()

    dataL = pd.DataFrame({"CellID": labelVec[0].tolist(), "cell_size": cellSizesVec})

    def _mk(df_arr):
        df = pd.DataFrame(df_arr)
        df.columns = clusterChannels
        return pd.concat((dataL, df), axis=1)

    dataL_full = _mk(dataCells)
    dataScaleSizeL_full = _mk(dataScaleSizeCells)
    dataCompenL_full = _mk(dataCompenCells)
    dataCompenScaleSizeL_full = _mk(dataCompenScaleSizeCells)

    # --- FIX: subset centroids to match keep mask ---
    cx = np.asarray(centroid_xs)[keep]
    cy = np.asarray(centroid_ys)[keep]

    for d in [dataL_full, dataScaleSizeL_full, dataCompenL_full, dataCompenScaleSizeL_full]:
        d["x_centroid"] = cx
        d["y_centroid"] = cy

    output_dir = pathlib.Path(output_dir).resolve()
    logging.info(f"Writing output files to {output_dir}")
    output_dir.mkdir(parents=True, exist_ok=True)

    dataScaleSizeL_full.to_csv(f"{output_dir}/single_cell_before_redsea.csv", index=False)
    dataCompenScaleSizeL_full.to_csv(f"{output_dir}/single_cell_after_redsea.csv", index=False)

    # Optional: return dataframes to inspect in notebook
    return dataScaleSizeL_full, dataCompenScaleSizeL_full


# Monkey-patch in this notebook session only:
rs.run_redsea = run_redsea_fixed
print("Patched redseapy.redsea.run_redsea for this session.")

Patched redseapy.redsea.run_redsea for this session.


In [3]:
import redseapy.redsea as rs

element_shape_code = 2 if ELEMENT_SHAPE.lower() == "star" else 1

before_df, after_df = rs.run_redsea(
    TIFF,
    SEG_MASK,
    MARKERS_CSV,
    OUTDIR,
    element_shape=element_shape_code,
    element_size=ELEMENT_SIZE,
    markers_of_interest=MARKERS_OF_INTEREST,
)

print("Done.")
print("Before:", before_df.shape, "After:", after_df.shape)
before_df.head()

2026-03-01 14:06:57,735 INFO: Reading image
2026-03-01 14:07:40,275 INFO: Reading segmentation mask
2026-03-01 16:43:56,350 INFO: Quantifying markers before correction
2026-03-01 16:44:16,262 INFO: Creating cell-cell contact matrix
2026-03-01 16:58:55,419 INFO: Performing correction


Progress: |██████████████████████████████████████████████████| 100.0% Complete


2026-03-01 18:15:20,448 INFO: Writing output files to /mnt/jwh83-data/Confetti/output/Redsea/20260127_ns444_CellSAM/B004-A-404


Done.
Before: (29873, 63) After: (29873, 63)


,CellID,cell_size,Hoechst1,MUC2,SOX9,MUC6,MUC1,GATA3,CD31,Synapto,...,CD154,Somatostatin,Ki67,CD49a,CD163,CD161,CD294,DRAQ5,x_centroid,y_centroid
0,1,1663.0,14722.858088,0.0,43.852676,0.183403,781.964522,819.464823,134.339747,493.052916,...,5.854480,672.371016,271.600120,2320.584486,1120.813590,216.052315,248.323512,4226.632592,1096.277811,130.276609
1,2,662.0,11487.726586,0.0,0.004532,0.000000,2055.904834,779.613293,1211.632931,456.214502,...,0.000000,75.929003,36.684290,4864.438066,11837.104230,2.250755,117.814199,2334.453172,1075.539275,274.061934
2,3,1246.0,9602.976726,0.0,7.247191,1.823435,705.456661,745.273676,8.735152,391.976726,...,13.005618,516.810594,210.582665,1186.518459,196.049759,139.460674,89.931782,3532.541734,961.096308,93.573034
3,4,583.0,5427.943396,0.0,65.178388,0.180103,483.960549,868.897084,6.010292,225.701544,...,3.300172,503.847341,164.559177,1321.274443,204.550600,176.735849,119.876501,1276.567753,1052.243568,346.051458
4,5,497.0,4272.450704,0.0,52.629779,0.633803,516.792757,795.066398,26.080483,244.199195,...,5.062374,502.655936,155.746479,1293.380282,199.832998,176.794769,118.939638,1101.977867,1030.267606,334.235412
